In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import httpx
from langchain_groq import ChatGroq

# Disable SSL verification to work around corporate proxy/firewall
client = httpx.Client(verify=False)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, http_client=client)

## Summarize messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about Lunapolis, the supposed capital of the moon, including its weather, population of cheese miners, and potential labor issues.\n\n## SUMMARY\nThe conversation history reveals that Lunapolis has extreme temperatures, with a high of 120C and a low of -100C. The city is home to 100,000 cheese miners. The cheese miners' union is likely to strike due to dissatisfaction with the new president. Notably, the initial premise of Lunapolis as the capital of the moon is unfounded, as the moon has no capital or permanent human settlements.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next steps should involve clarifying the factual inaccuracies regarding the moon's capital and exploring the fictional context of Lunapolis, if any. This could include discussing the origins of the concept of Lunapolis, its significance in a fictional setting, or s

In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, the supposed capital of the moon, including its weather, population of cheese miners, and potential labor issues.

## SUMMARY
The conversation history reveals that Lunapolis has extreme temperatures, with a high of 120C and a low of -100C. The city is home to 100,000 cheese miners. The cheese miners' union is likely to strike due to dissatisfaction with the new president. Notably, the initial premise of Lunapolis as the capital of the moon is unfounded, as the moon has no capital or permanent human settlements.

## ARTIFACTS
None

## NEXT STEPS
The next steps should involve clarifying the factual inaccuracies regarding the moon's capital and exploring the fictional context of Lunapolis, if any. This could include discussing the origins of the concept of Lunapolis, its significance in a fictional setting, or simply correcting the initial misunderstanding ab

## Trim/delete messages

In [7]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [8]:
agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='e4c2c196-4cfe-48c6-837f-29c04aeb1739'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='a3cabc9f-65d9-4d5f-b6e8-abb0dadfc3ec', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='1af13222-612b-456f-8e1f-77af088d75e5'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='28f1d719-bfdb-4925-a936-dca0c6ef852a', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='71bf75ed-08cf-4ca1-81b2-e20b37514953'),
              AIMessage(content="If the device has been exposed to extreme temperatures (very h

In [10]:
print(response["messages"][-1].content)

If the device has been exposed to extreme temperatures (very hot or cold), it may not turn on. Try to check if the device is at a normal room temperature. If it's been in a hot or cold environment, let it sit at room temperature for a while before trying to turn it on again.

Also, try the following basic troubleshooting steps:

1. Check the power cord and ensure it's properly connected to both the device and the power outlet.
2. Try pressing the power button for a longer duration (around 10-15 seconds) to see if it turns on.
3. If the device has a removable battery, try taking it out and putting it back in.
4. If none of the above steps work, try plugging the device into a different power outlet to rule out any issues with the electrical supply.

If the device still doesn't turn on, it may be a hardware issue, and you may need to contact the manufacturer or a professional for further assistance.
